## Analyze whether early or late snow changes more year to year or place to place.

* We know from previous notebooks that the value of `coef_2` corresponds to whether the snow season is early or late. 
* We want to study whether early/late season is more dependent on the year or on the location.
* We will use RMS Error to quantify the strength of these dependencies.

In [1]:
state='MA'
meas='SNWD'

In [2]:
import pandas as pd
import numpy as np
import urllib
import math

In [ ]:
%%time
%run lib/startup.py S

sc_type= S
10.34.165.155
namespace= grader-cse255-01
driver_host= 10.34.165.155


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/05/01 23:22:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/05/01 23:22:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
sparkContext= <SparkContext master=spark://spark-master-0.spark-headless.grader-cse255-01.svc.cluster.local:7077 appName=myapp>



/opt/bitnami/spark/python/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


version of ipwidgets= 8.1.6
parquet_root= /home/grader-cse255-01/public/Data/weather


In [ ]:
from matplotlib.pylab import *
import numpy as np
from lib.numpy_pack import packArray,unpackArray
from lib.spark_PCA import computeCov
from lib.computeStatistics import *

In [ ]:

decomp_parquet=parquet_root+'/weather-statistics/'+state+'-'+meas+'.parquet'
print('reading',decomp_parquet)
decomposition=sqlContext.read.parquet(decomp_parquet)
decomposition.count()

In [ ]:
decomposition.show(1)

### Recall interpretation of SNWD principal components
1. The amount of snow
2. The season being early/late
3. The season being long/short

In [ ]:
#create a table where colummns correspond to stations and rows to years
coeff="coeff_2"  ## Which coefficient to analyze
sqlContext.registerDataFrameAsTable(decomposition,'decomposition')
Columns='station, year, coeff_1,coeff_2,coeff_3,coeff_4,coeff_5'
Query="SELECT %s FROM decomposition"%Columns
print(Query)
pdf = sqlContext.sql(Query).toPandas()
year_station_table=pdf.pivot(index='year', columns='station', values=coeff)
print('shape of table=',year_station_table.shape)
year_station_table.tail(3)

In [ ]:
station_nulls=pd.isnull(year_station_table).mean(axis=0)
plot(sort(station_nulls))
ylabel('Fraction of years that are undefined')
xlabel('Number of stations')
grid()
title('fraction of undefined per station sorted in increasing order');

In [ ]:
year_nulls=pd.isnull(year_station_table).mean(axis=1)
year_nulls.plot();
grid()
ylabel('fraction of stations that are undefined')
title('fraction of undefined for each year');

In [ ]:
figure(figsize=(6, 6))
nulls=1-pd.isnull(year_station_table).T
imshow(nulls.values, cmap='Greys', aspect='auto')

xticks = np.arange(0, nulls.shape[1], 20)
yticks = np.arange(0, nulls.shape[0], 20)
plt.xticks(ticks=xticks, labels=nulls.columns[xticks], rotation=90)
plt.yticks(ticks=yticks, labels=nulls.index[yticks]);
grid();
title('Black = defined year and station');

### Different generations of stations
USC stations have been in operation for a long time, US1 stations started to come online in 1997

In [ ]:
from collections import Counter
L=[x[:3] for x in list(pdf['station'])]
Counter(L)

### Removing US1 stations
These stations are very recent. They might be in different areas and can bias our averages.

pdf2 is the cleaned table

In [ ]:
pdf2=pdf[pdf['station'].str.startswith("USC") | pdf['station'].str.startswith("USW")]
year_station_table=pdf2.pivot(index='year', columns='station', values=coeff)
print('year_station_table.shape=',year_station_table.shape)
year_station_table.tail(5)

## Studying the distribution of the coefficient

Coeff_1-3 capture most of the variance in the snow-depth distribution.

We can now look at how these coefficients vary from year to year.

In [ ]:
#coeff_1 corresponds to the overall amount of snow. 
# 2001,2011, 2015 were heavy snow years

PBox=pdf2[['year',coeff]][pdf2['year']>2000]
#PBox=pdf2[['year',coeff]][pdf2['year']<1900]
PBox.boxplot(by='year',figsize=[15,10]);
_mean=pdf2[coeff].mean()
print(f'Mean of {coeff} is {_mean}')
figure()
year_nulls=(1-pd.isnull(year_station_table)).sum(axis=1)
year_nulls[year_nulls.index>2000].plot();
#year_nulls[year_nulls.index<1900].plot();
grid()
title('number of stations defined per year');

## Statistical significance 
We use the t-test to evaluate the significance of the temperature being higher than the long tem mean.

In [ ]:
import pandas as pd
from scipy.stats import ttest_1samp

# Overall mean (from population or global data)
overall_mean = PBox[coeff].mean()  # Or some external value
# Choose a particular year to test
year_to_test = 2015
year_data = PBox[PBox['year'] == year_to_test][coeff]

# Perform one-sample t-test
t_stat, p_value = ttest_1samp(year_data, popmean=overall_mean)

print(f"T-statistic: {t_stat:.4f}, P-value: {p_value:.4g}")


### Estimating the effect of the year vs the effect of the station

To estimate the effect of time vs. location on the second eigenvector coefficient we
compute:

* The average row: `mean-by-station`
* The average column: `mean-by-year`

We then compute the RMS before and after subtracting either  the row or the column vector.

In [ ]:
def RMS(Mat):
    return np.sqrt(np.nanmean(Mat**2))

mean_by_year=np.nanmean(year_station_table,axis=1)
mean_by_station=np.nanmean(year_station_table,axis=0)
tbl_minus_year = (year_station_table.transpose()-mean_by_year).transpose()
tbl_minus_station = year_station_table-mean_by_station

print('total RMS                   = ',RMS(year_station_table))
print('RMS removing mean-by-station= ',RMS(tbl_minus_station),'reduction=',RMS(year_station_table)-RMS(tbl_minus_station))
print('RMS removing mean-by-year   = ',RMS(tbl_minus_year),'reduction=',RMS(year_station_table)-RMS(tbl_minus_year))

### Conclusion Of Analysis
The effect of time is about four times as large as the effect of location.

In [ ]:
years=list(year_station_table.index)
plot(years,mean_by_year)
grid()
title('mean as a function of year')

## mean is constat over time over time

In [ ]:
M=np.array(mean_by_year)
l=int(M.shape[0]/2)
group1=M[0:l]
group2=M[l:]

In [ ]:
from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(group1, group2, equal_var=False)  # Welch's t-test
print(f't_stat={t_stat}, p_value={p_value})')

### What is the conclusion ?

## Summary
* The problem of missing data is prevalent and needs to be addressed.
* RMS can be used to quantify the effect of different factors (here, time vs. space)
* The snow season in NY seems to be getting earlier and earlier since 1960.
* but the high variance places doubt on this conclusion.